# ConsDB Camera Rotator Angle Survey

**Author:** Aaron Roodman  
**Date Created:** 2026-09-02  
**Last Modified:** 2026-09-02  
**Status:** Draft  
**Keywords:** camera rotator, physical_rotator_angle, ConsDB, visit1_quicklook, missing data, visit1_efd  

## Description

Survey **many nights** of LSSTCam visits using **ConsDB only** — no Butler and no
EFD client — to characterise *how* and *when* ConsDB is missing
`visit1_quicklook.physical_rotator_angle`.

Reading the raw `visitInfo` through the Butler is far too slow for a multi-night
survey (it touches raw files per visit), so everything here comes from ConsDB
queries, which are fast and can cover months in a few chunked queries.

The central distinction this notebook draws is **two different kinds of
"missing"**:

1. **No `visit1_quicklook` row at all** — nothing was ever aggregated for the
   visit, so *every* quicklook column is absent, not just the rotator.
2. **Row present, `physical_rotator_angle` IS NULL** — the visit was processed
   but this particular column was not filled.

These have completely different causes and different fixes, so the notebook
separates them everywhere rather than reporting one "missing" number.

Key functionality:
1. Chunked ConsDB query of `visit1` LEFT JOIN `visit1_quicklook` over a night
   range, with an on-disk parquet cache so re-runs are instant.
2. Split the missing visits into the two categories above.
3. **When:** per-night missing fraction time series, plus breakdowns by
   `img_type`, `science_program`, and `band`; detect whole-night outages and
   whether missingness starts or stops on a particular date.
4. **How:** for row-present-but-NULL visits, check which *other* quicklook
   column families were populated (science pipeline, AOS Zernikes, guider) to
   localise which producer did not run.
5. Structure in time — are missing visits contiguous runs (a service outage) or
   scattered singletons? Where do they sit within a night?
6. **Recoverability without the Butler:** cross-match against
   `efd_lsstcam.visit1_efd.mt_pointing_mount_position_rotator_mean`, a rotator
   angle already in ConsDB, and report how many missing visits it recovers.

**Output:** Summary tables printed inline, six diagnostic panels, and an
optional parquet of the missing visits.

**Companion notebook:** `camera_rotator_angle_check.ipynb` does the per-night
three-source consistency check (ConsDB vs EFD vs Butler `visitInfo`).  This one
is the wide, fast, ConsDB-only view.

**Note:** Self-contained — only `lsst.summit.utils` plus standard scientific
Python, no personal repository code, so it can be shared with Rubin colleagues.

## ConsDB columns used

| Table | Column | Role |
|---|---|---|
| `cdb_*.visit1` | `img_type`, `science_program`, `band`, `obs_start`, `altitude` | visit classification / when |
| `cdb_*.visit1_quicklook` | `physical_rotator_angle` | the quantity under study |
| `cdb_*.visit1_quicklook` | `visit_id` | presence marker — NULL means no row |
| `cdb_*.visit1_quicklook` | `n_inputs`, `psf_sigma_median`, `z4`, `guider_psf_fwhm` | which producer populated the row |
| `efd_*.visit1_efd` | `mt_pointing_mount_position_rotator_mean` | Butler-free recovery |


## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-09-02 | Aaron Roodman | Initial version |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
   - [4.1 visit1 + visit1_quicklook](#data-visits)
   - [4.2 ConsDB-native rotator from visit1_efd](#data-efd)
5. [Analysis](#analysis)
   - [5.1 How: no row vs NULL column](#analysis-how)
   - [5.2 When: per night](#analysis-when)
   - [5.3 By img_type, program, band](#analysis-cats)
   - [5.4 Which producer populated the row](#analysis-producer)
   - [5.5 Structure in time](#analysis-runs)
   - [5.6 Recoverability without the Butler](#analysis-recover)
6. [Results & Plots](#results)


<a id='params'></a>
## Parameters


In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_min = 20250601        # first night of the survey (YYYYMMDD)
day_obs_max = 20260901        # last night of the survey (inclusive)

instrument = 'lsstcam'        # ConsDB schema suffix: 'lsstcam' | 'lsstcomcam'
consdb_url = 'auto'           # 'auto' picks in-pod vs external endpoint

chunk_nights = 30             # nights per ConsDB query (keeps each query small)
query_efd_rotator = True      # also pull efd_*.visit1_efd for recoverability

# Cache: the survey query is the slow step, so results are cached to parquet
use_cache = True
cache_dir = 'output/rotator_survey_cache'

# Visits to exclude from the "should have a rotator angle" denominator.
# Dome-closed calibrations have no meaningful sky pointing or rotator demand.
calib_img_types = ['bias', 'dark', 'flat']

output_dir = 'output'
output_file = None            # e.g. 'rotator_missing.parquet'; None = no file


<a id='setup'></a>
## Setup & Imports


In [ ]:
import os
import warnings
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from lsst.summit.utils import ConsDbClient

from tqdm.notebook import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 11,
    'legend.frameon': False,
})

print(f'Survey window: {day_obs_min} .. {day_obs_max}   instrument: {instrument}')


<a id='functions'></a>
## Helper Functions


In [ ]:
IN_POD_CONSDB_URL = 'http://consdb-pq.consdb:8080/consdb'
EXTERNAL_CONSDB_URL = 'https://usdf-rsp.slac.stanford.edu/consdb'


def in_rsp():
    """True when running inside an RSP (Nublado) JupyterLab pod."""
    return os.path.isdir('/etc/nublado')


def make_consdb_client(url='auto', token_file=None):
    """Return a `lsst.summit.utils.ConsDbClient`.

    ``url='auto'`` uses the in-pod host inside the RSP and the public
    token-injected endpoint elsewhere.  The in-pod host must bypass the HTTP
    proxy (else 502), so ``.consdb`` is appended to ``$no_proxy``; the external
    endpoint needs an RSP token from ``~/.lsst/consdb_token`` or
    ``$ACCESS_TOKEN``.

    Parameters
    ----------
    url : `str`
        ConsDB endpoint, or ``'auto'``.
    token_file : `str`, optional
        Path to a file holding an RSP access token.

    Returns
    -------
    `lsst.summit.utils.ConsDbClient`
    """
    if url == 'auto':
        url = IN_POD_CONSDB_URL if in_rsp() else EXTERNAL_CONSDB_URL
    no_proxy = os.environ.get('no_proxy', '')
    if '.consdb' not in no_proxy:
        os.environ['no_proxy'] = (no_proxy + ',.consdb') if no_proxy else '.consdb'
    if '@' not in url and 'consdb-pq.consdb' not in url:
        tf = Path(token_file) if token_file else Path.home() / '.lsst' / 'consdb_token'
        token = tf.read_text().strip() if tf.exists() else os.environ.get('ACCESS_TOKEN')
        if token:
            url = url.replace('://', f'://user:{token}@', 1)
    return ConsDbClient(url)


In [ ]:
def day_obs_to_date(day_obs):
    """YYYYMMDD integer -> `datetime.date`."""
    s = str(int(day_obs))
    return date(int(s[0:4]), int(s[4:6]), int(s[6:8]))


def date_to_day_obs(d):
    """`datetime.date` -> YYYYMMDD integer."""
    return d.year * 10000 + d.month * 100 + d.day


def day_obs_chunks(day_obs_min, day_obs_max, chunk_nights=30):
    """Split a night range into ``(lo, hi)`` day_obs pairs of at most
    ``chunk_nights`` calendar nights each.

    Chunking walks real calendar dates, so the bounds never land on impossible
    day_obs values such as 20260732.  Because YYYYMMDD integers sort in date
    order, each pair is usable directly in a SQL ``BETWEEN``.

    Parameters
    ----------
    day_obs_min, day_obs_max : `int`
        Inclusive night range as YYYYMMDD.
    chunk_nights : `int`
        Maximum nights per chunk.

    Returns
    -------
    `list` [(`int`, `int`)]
        Inclusive ``(lo, hi)`` day_obs bounds covering the range in order.
    """
    if chunk_nights < 1:
        raise ValueError('chunk_nights must be >= 1')
    lo, hi = day_obs_to_date(day_obs_min), day_obs_to_date(day_obs_max)
    if hi < lo:
        raise ValueError('day_obs_max is before day_obs_min')
    out, cur = [], lo
    while cur <= hi:
        end = min(cur + timedelta(days=chunk_nights - 1), hi)
        out.append((date_to_day_obs(cur), date_to_day_obs(end)))
        cur = end + timedelta(days=1)
    return out


In [ ]:
# Quicklook columns used as "which producer populated this row" probes.
# Each belongs to a different family that is written by a different task, so the
# pattern of which are non-NULL localises what did and did not run for a visit.
PRODUCER_PROBES = {
    'n_inputs': 'visit aggregation (n CCDs)',
    'psf_sigma_median': 'science pipeline PSF',
    'z4': 'AOS Zernikes',
    'guider_psf_fwhm': 'guider',
}

VISIT_COLS = ['visit_id', 'day_obs', 'seq_num', 'img_type', 'science_program',
              'observation_reason', 'band', 'target_name', 'altitude', 'azimuth',
              'sky_rotation', 'exp_time', 'obs_start']


def fetch_visits(cdb, day_obs_min, day_obs_max, instrument='lsstcam',
                 chunk_nights=30):
    """`visit1` LEFT JOIN `visit1_quicklook` over a night range, chunked.

    The join is LEFT so visits with **no quicklook row at all** still appear;
    those are one of the two flavours of "missing" this notebook separates.
    ``ql_visit_id`` is the presence marker: NULL there means no row exists,
    whereas a non-NULL ``ql_visit_id`` with a NULL ``physical_rotator_angle``
    means the row exists but the column was not filled.

    Parameters
    ----------
    cdb : `lsst.summit.utils.ConsDbClient`
        ConsDB client.
    day_obs_min, day_obs_max : `int`
        Inclusive night range, YYYYMMDD.
    instrument : `str`
        ConsDB schema suffix.
    chunk_nights : `int`
        Nights per query.

    Returns
    -------
    `pandas.DataFrame`
        One row per visit, sorted by ``(day_obs, seq_num)``.
    """
    vsel = ', '.join(f'v.{c}' for c in VISIT_COLS)
    psel = ', '.join(f'ql.{c}' for c in PRODUCER_PROBES)
    chunks = day_obs_chunks(day_obs_min, day_obs_max, chunk_nights)
    frames = []
    for lo, hi in tqdm(chunks, desc='visit1 chunks'):
        query = f"""
            SELECT {vsel},
                   ql.visit_id AS ql_visit_id,
                   ql.physical_rotator_angle,
                   {psel}
            FROM cdb_{instrument}.visit1 AS v
            LEFT JOIN cdb_{instrument}.visit1_quicklook AS ql
                ON ql.visit_id = v.visit_id
            WHERE v.day_obs BETWEEN {lo} AND {hi}
            ORDER BY v.day_obs, v.seq_num
        """
        try:
            frames.append(cdb.query(query).to_pandas())
        except Exception as exc:
            print(f'  chunk {lo}-{hi} FAILED: {type(exc).__name__}: {exc}')
    if not frames:
        raise RuntimeError('no ConsDB chunks succeeded')
    df = pd.concat(frames, ignore_index=True)

    num = (['physical_rotator_angle', 'sky_rotation', 'altitude', 'azimuth',
            'exp_time'] + list(PRODUCER_PROBES))
    for col in num:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)


def fetch_efd_rotator(cdb, day_obs_min, day_obs_max, instrument='lsstcam',
                      chunk_nights=30):
    """ConsDB-native rotator angle from the transformed-EFD ``visit1_efd`` table.

    ``mt_pointing_mount_position_rotator_mean`` is the mean rotator axis position
    reported by the rotator component, already summarised per visit in ConsDB —
    so it needs neither the Butler nor a live EFD client, which is what makes it
    usable across a whole survey.  Queried separately from `visit1` (rather than
    as a third JOIN) so a failure or an absent schema degrades gracefully.

    Returns
    -------
    `pandas.DataFrame`
        Columns ``visit_id``, ``efd_rotator_angle``, ``efd_rotator_std``.
        Empty (but correctly typed) if the table is unavailable.
    """
    cols = ['visit_id', 'day_obs', 'seq_num',
            'mt_pointing_mount_position_rotator_mean',
            'mt_pointing_mount_position_rotator_stddev']
    out_cols = ['visit_id', 'efd_rotator_angle', 'efd_rotator_std']
    frames = []
    for lo, hi in tqdm(day_obs_chunks(day_obs_min, day_obs_max, chunk_nights),
                       desc='visit1_efd chunks'):
        query = f"""
            SELECT {', '.join(cols)}
            FROM efd_{instrument}.visit1_efd
            WHERE day_obs BETWEEN {lo} AND {hi}
        """
        try:
            frames.append(cdb.query(query).to_pandas())
        except Exception as exc:
            print(f'  visit1_efd chunk {lo}-{hi} FAILED: '
                  f'{type(exc).__name__}: {exc}')
    if not frames:
        print('  visit1_efd unavailable — recoverability section will be empty')
        return pd.DataFrame(columns=out_cols)
    df = pd.concat(frames, ignore_index=True).rename(columns={
        'mt_pointing_mount_position_rotator_mean': 'efd_rotator_angle',
        'mt_pointing_mount_position_rotator_stddev': 'efd_rotator_std'})
    for c in ('efd_rotator_angle', 'efd_rotator_std'):
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df[out_cols].drop_duplicates('visit_id')


In [ ]:
MISSING_LABELS = {
    'present': 'physical_rotator_angle present',
    'null_column': 'quicklook row exists, column is NULL',
    'no_row': 'no visit1_quicklook row at all',
}


def classify_missing(df):
    """Label every visit ``present`` / ``null_column`` / ``no_row``.

    This is the notebook's core distinction.  A NULL ``physical_rotator_angle``
    can mean two very different things, and lumping them into a single "missing"
    count hides the cause:

    * ``no_row`` — no ``visit1_quicklook`` row exists, so nothing at all was
      aggregated for the visit.
    * ``null_column`` — the row is there but this column was not filled.

    Parameters
    ----------
    df : `pandas.DataFrame`
        Must have ``ql_visit_id`` and ``physical_rotator_angle``.

    Returns
    -------
    `pandas.Series` of `str`
        The per-visit category, aligned to ``df.index``.
    """
    has_row = df['ql_visit_id'].notna()
    has_val = df['physical_rotator_angle'].notna()
    return pd.Series(
        np.select([has_val, has_row & ~has_val], ['present', 'null_column'],
                  default='no_row'),
        index=df.index, name='rot_status')


def missing_rate_table(df, by, status_col='rot_status', min_count=1):
    """Missing counts and rates grouped by one or more columns.

    Parameters
    ----------
    df : `pandas.DataFrame`
        Must contain ``status_col``.
    by : `str` or `list` [`str`]
        Column(s) to group by; NaNs become the string ``'(none)'``.
    status_col : `str`
        Column holding the `classify_missing` labels.
    min_count : `int`
        Drop groups with fewer than this many visits.

    Returns
    -------
    `pandas.DataFrame`
        Columns ``n``, ``present``, ``null_column``, ``no_row``,
        ``missing``, ``miss_frac``; sorted by ``missing`` descending.
    """
    keys = [by] if isinstance(by, str) else list(by)
    g = df.copy()
    for k in keys:
        g[k] = g[k].fillna('(none)') if g[k].dtype == object else g[k]
    tab = (g.groupby(keys)[status_col].value_counts().unstack(fill_value=0))
    for c in ('present', 'null_column', 'no_row'):
        if c not in tab.columns:
            tab[c] = 0
    tab = tab[['present', 'null_column', 'no_row']]
    tab['n'] = tab.sum(axis=1)
    tab['missing'] = tab['null_column'] + tab['no_row']
    tab['miss_frac'] = tab['missing'] / tab['n']
    tab = tab[tab['n'] >= min_count]
    return tab[['n', 'present', 'null_column', 'no_row', 'missing',
                'miss_frac']].sort_values('missing', ascending=False)


def missing_runs(df, status_col='rot_status'):
    """Contiguous runs of missing visits within each night, by ``seq_num``.

    A handful of long runs points at a service outage or an unprocessed block;
    many length-1 runs point at per-visit dropouts.  The two call for different
    follow-up, so the run-length distribution is the discriminator.

    Parameters
    ----------
    df : `pandas.DataFrame`
        Must have ``day_obs``, ``seq_num`` and ``status_col``.
    status_col : `str`
        Column holding the `classify_missing` labels.

    Returns
    -------
    `pandas.DataFrame`
        One row per run: ``day_obs``, ``seq_start``, ``seq_end``, ``length``,
        ``status`` (the categories present in the run, joined by ``'+'``).
    """
    rows = []
    for day, sub in df.sort_values(['day_obs', 'seq_num']).groupby('day_obs'):
        miss = sub[sub[status_col] != 'present']
        if miss.empty:
            continue
        # A new run starts wherever this missing visit is not the immediately
        # following seq_num of the previous missing visit.
        seq = miss['seq_num'].to_numpy()
        brk = np.r_[True, np.diff(seq) != 1]
        gids = np.cumsum(brk)
        for gid in np.unique(gids):
            run = miss[gids == gid]
            rows.append({'day_obs': day,
                         'seq_start': int(run['seq_num'].iloc[0]),
                         'seq_end': int(run['seq_num'].iloc[-1]),
                         'length': len(run),
                         'status': '+'.join(sorted(run[status_col].unique()))})
    return pd.DataFrame(rows, columns=['day_obs', 'seq_start', 'seq_end',
                                       'length', 'status'])


<a id='data'></a>
## Data Access


<a id='data-visits'></a>
### 4.1 `visit1` + `visit1_quicklook`

Chunked over the night range and cached to parquet, so re-running the analysis
below costs nothing.  Delete the cache file (or set `use_cache = False`) to
re-query.


In [ ]:
Path(cache_dir).mkdir(parents=True, exist_ok=True)
cache_visits = Path(cache_dir) / f'visits_{instrument}_{day_obs_min}_{day_obs_max}.parquet'

if use_cache and cache_visits.exists():
    visits = pd.read_parquet(cache_visits)
    print(f'Loaded {len(visits)} visits from cache: {cache_visits}')
else:
    cdb = make_consdb_client(consdb_url)
    n_chunks = len(day_obs_chunks(day_obs_min, day_obs_max, chunk_nights))
    print(f'Querying ConsDB in {n_chunks} chunk(s) of <= {chunk_nights} nights...')
    visits = fetch_visits(cdb, day_obs_min, day_obs_max,
                          instrument=instrument, chunk_nights=chunk_nights)
    visits.to_parquet(cache_visits, index=False)
    print(f'Fetched {len(visits)} visits; cached to {cache_visits}')

nights = sorted(visits.day_obs.unique())
print(f'\n{len(visits)} visits over {len(nights)} nights '
      f'({nights[0]} .. {nights[-1]})')
print(f'img_type values: {sorted(visits.img_type.fillna("(none)").unique())}')
visits.head()


<a id='data-efd'></a>
### 4.2 ConsDB-native rotator from `visit1_efd`

`efd_*.visit1_efd.mt_pointing_mount_position_rotator_mean` is a per-visit rotator
angle **already in ConsDB**, so it is a Butler-free, EFD-client-free recovery
path — the whole reason this survey can cover months.


In [ ]:
cache_efd = Path(cache_dir) / f'efd_{instrument}_{day_obs_min}_{day_obs_max}.parquet'

if not query_efd_rotator:
    efd = pd.DataFrame(columns=['visit_id', 'efd_rotator_angle', 'efd_rotator_std'])
    print('query_efd_rotator = False — skipping visit1_efd')
elif use_cache and cache_efd.exists():
    efd = pd.read_parquet(cache_efd)
    print(f'Loaded {len(efd)} visit1_efd rows from cache')
else:
    cdb = make_consdb_client(consdb_url)
    efd = fetch_efd_rotator(cdb, day_obs_min, day_obs_max,
                            instrument=instrument, chunk_nights=chunk_nights)
    efd.to_parquet(cache_efd, index=False)
    print(f'Fetched {len(efd)} visit1_efd rows; cached')

vis = visits.merge(efd, on='visit_id', how='left')
assert len(vis) == len(visits), 'visit1_efd merge changed the row count'
vis['rot_status'] = classify_missing(vis)

# Calibration frames are dome-closed: no meaningful rotator demand, so they are
# tracked separately rather than inflating the missing-rate denominator.
vis['is_calib'] = vis.img_type.fillna('').str.lower().isin(
    [t.lower() for t in calib_img_types])
print(f'\n{len(vis)} visits  |  on-sky {int((~vis.is_calib).sum())}  '
      f'calib {int(vis.is_calib.sum())}')
print(f'visit1_efd rotator available for '
      f'{int(vis.efd_rotator_angle.notna().sum())}/{len(vis)} visits')


<a id='analysis'></a>
## Analysis


<a id='analysis-how'></a>
### 5.1 How: no row vs NULL column

The headline split.  These two categories have different causes, so every later
breakdown keeps them separate.


In [ ]:
def status_summary(df, label):
    """Print the present / null_column / no_row breakdown for a subset."""
    n = len(df)
    print(f'\n{label}  (n = {n})')
    print('-' * 68)
    if n == 0:
        print('  (empty)')
        return
    vc = df.rot_status.value_counts()
    for key in ('present', 'null_column', 'no_row'):
        c = int(vc.get(key, 0))
        print(f'  {MISSING_LABELS[key]:<44} {c:7d}  {100.0 * c / n:6.2f}%')
    miss = n - int(vc.get('present', 0))
    print(f'  {"TOTAL missing":<44} {miss:7d}  {100.0 * miss / n:6.2f}%')

status_summary(vis, 'All visits')
status_summary(vis[~vis.is_calib], 'On-sky visits (excluding '
                                   f'{calib_img_types})')
status_summary(vis[vis.is_calib], f'Calibration visits ({calib_img_types})')


<a id='analysis-when'></a>
### 5.2 When: per night

Per-night missing fractions expose whole-night outages and any date at which the
behaviour changes (a service deployment or a schema change).


In [ ]:
by_night = missing_rate_table(vis, 'day_obs').sort_index()

full_out = by_night[by_night.miss_frac >= 0.999]
partial = by_night[(by_night.missing > 0) & (by_night.miss_frac < 0.999)]
clean = by_night[by_night.missing == 0]

print(f'Nights surveyed          : {len(by_night)}')
print(f'  fully missing (100%)   : {len(full_out)}')
print(f'  partially missing      : {len(partial)}')
print(f'  complete (0% missing)  : {len(clean)}')

if len(full_out):
    print(f'\nWhole-night outages ({len(full_out)} nights):')
    print(full_out[['n', 'null_column', 'no_row']].to_string())

# Does the behaviour change at a particular date?
present_nights = by_night[by_night.present > 0].index
missing_nights = by_night[by_night.missing > 0].index
if len(present_nights):
    print(f'\nFirst / last night WITH a rotator angle : '
          f'{present_nights.min()} / {present_nights.max()}')
if len(missing_nights):
    print(f'First / last night WITH a missing angle : '
          f'{missing_nights.min()} / {missing_nights.max()}')

print('\nWorst 15 nights by missing count:')
print(by_night.sort_values('missing', ascending=False).head(15).to_string(
    float_format=lambda v: f'{v:.3f}'))


<a id='analysis-cats'></a>
### 5.3 By `img_type`, `science_program`, `band`

Which kinds of visit are affected — this usually identifies the responsible
producer faster than the time series does.


In [ ]:
for key in ('img_type', 'science_program', 'band', 'observation_reason'):
    if key not in vis.columns:
        continue
    tab = missing_rate_table(vis, key, min_count=1)
    print(f'\n=== missing by {key} ===')
    print(tab.head(20).to_string(float_format=lambda v: f'{v:.3f}'))

# img_type x status, on-sky only, as rates
print('\n=== on-sky visits only, missing rate by img_type ===')
print(missing_rate_table(vis[~vis.is_calib], 'img_type').to_string(
    float_format=lambda v: f'{v:.3f}'))


<a id='analysis-producer'></a>
### 5.4 Which producer populated the row

For `null_column` visits the quicklook row exists, so the pattern of *other*
populated columns says which task ran and which did not.  If those visits have
science-pipeline and AOS columns filled but no rotator angle, the gap is
specific to whatever writes `physical_rotator_angle`; if the row is empty across
the board, the row was merely created and never filled.


In [ ]:
probe_cols = [c for c in PRODUCER_PROBES if c in vis.columns]

print(f'Fraction of visits with each probe column populated, by rot_status:')
print('(probe columns come from different producers writing the same row)\n')
rows = []
for status in ('present', 'null_column', 'no_row'):
    sub = vis[vis.rot_status == status]
    if not len(sub):
        continue
    rows.append({'rot_status': status, 'n': len(sub),
                 **{c: sub[c].notna().mean() for c in probe_cols}})
prod = pd.DataFrame(rows).set_index('rot_status')
print(prod.to_string(float_format=lambda v: f'{v:.3f}'))

print('\nProbe column meanings:')
for c in probe_cols:
    print(f'  {c:<20} {PRODUCER_PROBES[c]}')

nullcol = vis[vis.rot_status == 'null_column']
if len(nullcol):
    n_all_empty = int((~nullcol[probe_cols].notna().any(axis=1)).sum())
    n_some = len(nullcol) - n_all_empty
    print(f'\nOf {len(nullcol)} null_column visits:')
    print(f'  {n_all_empty} have NO probe column populated '
          f'(row created but never filled)')
    print(f'  {n_some} have at least one probe populated '
          f'(gap specific to physical_rotator_angle)')
else:
    print('\nNo null_column visits — every missing visit lacks the row entirely.')


<a id='analysis-runs'></a>
### 5.5 Structure in time

Contiguous runs of missing visits within a night distinguish an outage (a few
long runs) from per-visit dropouts (many length-1 runs).


In [ ]:
runs = missing_runs(vis)
print(f'{len(runs)} contiguous missing runs across {runs.day_obs.nunique() if len(runs) else 0} nights')

if len(runs):
    print(f'\nRun length: median {runs.length.median():.0f}, '
          f'mean {runs.length.mean():.1f}, max {runs.length.max()}')
    print(f'  singletons (length 1): {int((runs.length == 1).sum())}'
          f'  ({100.0 * (runs.length == 1).mean():.1f}% of runs)')
    print(f'  runs >= 10 visits    : {int((runs.length >= 10).sum())}')
    print(f'\nTotal missing visits in runs >= 10: '
          f'{int(runs.loc[runs.length >= 10, "length"].sum())} '
          f'of {int(runs.length.sum())}')
    print('\nLongest 15 runs:')
    print(runs.sort_values('length', ascending=False).head(15).to_string(index=False))

    # Where in the night do missing visits sit? Normalised seq_num position.
    pos = []
    for day, sub in vis.groupby('day_obs'):
        lo, hi = sub.seq_num.min(), sub.seq_num.max()
        if hi > lo:
            m = sub.rot_status != 'present'
            pos.extend(((sub.loc[m, 'seq_num'] - lo) / (hi - lo)).tolist())
    if pos:
        pos = np.asarray(pos)
        print(f'\nPosition of missing visits within their night '
              f'(0 = first, 1 = last), n={len(pos)}:')
        for lo_e, hi_e in [(0, .2), (.2, .4), (.4, .6), (.6, .8), (.8, 1.01)]:
            f = float(((pos >= lo_e) & (pos < hi_e)).mean())
            print(f'  {lo_e:.1f}-{min(hi_e, 1.0):.1f}: {f * 100:5.1f}%  '
                  f'{"#" * int(round(f * 50))}')


<a id='analysis-recover'></a>
### 5.6 Recoverability without the Butler

How many of the missing visits can be filled from
`visit1_efd.mt_pointing_mount_position_rotator_mean` — a pure ConsDB lookup.
The agreement check on visits where *both* exist validates the substitution
before it is trusted for the ones where only the EFD value exists.


In [ ]:
def wrap180(x):
    """Wrap an angle in degrees to [-180, 180]."""
    out = (np.asarray(x, dtype=float) + 180.0) % 360.0 - 180.0
    return float(out) if np.ndim(x) == 0 else out

miss = vis[vis.rot_status != 'present']
print(f'Missing visits                         : {len(miss)}')
if len(miss):
    n_rec = int(miss.efd_rotator_angle.notna().sum())
    print(f'  recoverable from visit1_efd          : {n_rec}  '
          f'({100.0 * n_rec / len(miss):.1f}%)')
    print(f'  still unrecoverable via ConsDB       : {len(miss) - n_rec}')
    for status in ('null_column', 'no_row'):
        s = miss[miss.rot_status == status]
        if len(s):
            r = int(s.efd_rotator_angle.notna().sum())
            print(f'    {status:<12}: {r}/{len(s)} recoverable '
                  f'({100.0 * r / len(s):.1f}%)')

# Validate the substitution where both values exist
both = vis[vis.physical_rotator_angle.notna() & vis.efd_rotator_angle.notna()]
print(f'\nAgreement check on {len(both)} visits having BOTH values:')
if len(both):
    d = wrap180(both.efd_rotator_angle - both.physical_rotator_angle)
    print(f'  visit1_efd - physical_rotator_angle: mean {np.mean(d):+.4f} deg, '
          f'std {np.std(d):.4f} deg')
    print(f'  median {np.median(d):+.4f}, '
          f'95th pct |d| {np.percentile(np.abs(d), 95):.4f}, '
          f'max |d| {np.max(np.abs(d)):.4f} deg')
    n_bad = int((np.abs(d) > 0.5).sum())
    print(f'  |diff| > 0.5 deg: {n_bad}/{len(both)} '
          f'({100.0 * n_bad / len(both):.2f}%)')
    vis.loc[both.index, 'efd_minus_consdb'] = d
else:
    print('  none — cannot validate the substitution')

vis['rotator_best'] = vis.physical_rotator_angle.fillna(vis.efd_rotator_angle)
vis['rotator_best_source'] = np.select(
    [vis.physical_rotator_angle.notna(), vis.efd_rotator_angle.notna()],
    ['consdb_quicklook', 'consdb_visit1_efd'], default='none')
print('\nBest available rotator angle, by source:')
print(vis.rotator_best_source.value_counts().to_string())


<a id='results'></a>
## Results & Plots


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 13))

night_idx = np.arange(len(by_night))
labels = [str(d) for d in by_night.index]

def sparse_ticks(ax, labels, nmax=12):
    """Label at most nmax nights so the axis stays readable."""
    step = max(1, len(labels) // nmax)
    ax.set_xticks(np.arange(len(labels))[::step])
    ax.set_xticklabels(labels[::step], rotation=45, ha='right', fontsize=7)

# (a) per-night missing fraction, split by category
ax = axes[0, 0]
ax.bar(night_idx, by_night.null_column / by_night.n, width=1.0,
       label='row exists, column NULL')
ax.bar(night_idx, by_night.no_row / by_night.n, width=1.0,
       bottom=by_night.null_column / by_night.n, label='no quicklook row')
ax.set_ylabel('fraction of visits missing')
ax.set_title('(a) Missing fraction per night')
ax.set_ylim(0, 1.02)
sparse_ticks(ax, labels)
ax.legend(loc='best', fontsize=8)

# (b) per-night visit counts, present vs missing
ax = axes[0, 1]
ax.bar(night_idx, by_night.present, width=1.0, label='present')
ax.bar(night_idx, by_night.missing, width=1.0, bottom=by_night.present,
       label='missing')
ax.set_ylabel('visits')
ax.set_title('(b) Visits per night (present vs missing)')
sparse_ticks(ax, labels)
ax.legend(loc='best', fontsize=8)

# (c) missing rate by img_type
ax = axes[1, 0]
t = missing_rate_table(vis, 'img_type').sort_values('n', ascending=True)
y = np.arange(len(t))
ax.barh(y, t.null_column / t.n, label='column NULL')
ax.barh(y, t.no_row / t.n, left=t.null_column / t.n, label='no row')
ax.set_yticks(y)
ax.set_yticklabels([f'{i} (n={n})' for i, n in zip(t.index, t.n)], fontsize=8)
ax.set_xlabel('fraction missing')
ax.set_title('(c) Missing rate by img_type')
ax.set_xlim(0, 1.02)
ax.legend(loc='best', fontsize=8)

# (d) missing rate by science_program (top 12 by visit count)
ax = axes[1, 1]
t = missing_rate_table(vis, 'science_program').sort_values(
    'n', ascending=False).head(12).sort_values('n')
y = np.arange(len(t))
ax.barh(y, t.miss_frac)
ax.set_yticks(y)
ax.set_yticklabels([f'{str(i)[:26]} (n={n})' for i, n in zip(t.index, t.n)],
                   fontsize=7)
ax.set_xlabel('fraction missing')
ax.set_title('(d) Missing rate by science_program (top 12 by n)')
ax.set_xlim(0, 1.02)

# (e) run-length distribution
ax = axes[2, 0]
if len(runs):
    bins = np.arange(0.5, min(runs.length.max(), 60) + 1.5)
    ax.hist(runs.length.clip(upper=60), bins=bins)
    ax.set_yscale('log')
    ax.set_xlabel('contiguous missing run length [visits]')
    ax.set_ylabel('runs')
    ax.set_title(f'(e) Missing-run lengths (n={len(runs)} runs)')
else:
    ax.text(0.5, 0.5, 'no missing visits', ha='center', va='center',
            transform=ax.transAxes)
    ax.set_title('(e) Missing-run lengths')

# (f) recoverability + agreement
ax = axes[2, 1]
if len(both):
    d = wrap180(both.efd_rotator_angle - both.physical_rotator_angle)
    ax.hist(np.clip(d, -1, 1), bins=60)
    ax.set_yscale('log')
    ax.set_xlabel('visit1_efd - physical_rotator_angle [deg], clipped to +/-1')
    ax.set_ylabel('visits')
    ax.set_title(f'(f) ConsDB-native rotator agreement (n={len(both)}, '
                 f'std {np.std(d):.3f} deg)')
else:
    ax.text(0.5, 0.5, 'no overlapping visits', ha='center', va='center',
            transform=ax.transAxes)
    ax.set_title('(f) ConsDB-native rotator agreement')

fig.suptitle(f'ConsDB physical_rotator_angle survey — {instrument}, '
             f'{day_obs_min}..{day_obs_max}, {len(vis)} visits, '
             f'{len(by_night)} nights', fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Summary
# ============================================================
n = len(vis)
n_miss = int((vis.rot_status != 'present').sum())
onsky = vis[~vis.is_calib]

print('=' * 74)
print(f'ConsDB physical_rotator_angle survey — {instrument}')
print(f'{day_obs_min} .. {day_obs_max}   {n} visits over {len(by_night)} nights')
print('=' * 74)

print(f'\nMissing overall            : {n_miss}/{n} ({100.0 * n_miss / max(n, 1):.2f}%)')
print(f'  no quicklook row         : {int((vis.rot_status == "no_row").sum())}')
print(f'  row exists, column NULL  : {int((vis.rot_status == "null_column").sum())}')
if len(onsky):
    n_om = int((onsky.rot_status != 'present').sum())
    print(f'Missing, on-sky visits only: {n_om}/{len(onsky)} '
          f'({100.0 * n_om / len(onsky):.2f}%)')

print(f'\nNights fully missing       : {len(full_out)}')
print(f'Nights partially missing   : {len(partial)}')
print(f'Nights complete            : {len(clean)}')

if len(runs):
    print(f'\nMissing structure          : {len(runs)} runs, '
          f'{int((runs.length == 1).sum())} singletons, '
          f'longest {runs.length.max()} visits')

if len(miss):
    n_rec = int(miss.efd_rotator_angle.notna().sum())
    print(f'\nRecoverable from ConsDB visit1_efd : {n_rec}/{len(miss)} '
          f'({100.0 * n_rec / len(miss):.1f}%) — no Butler needed')
if len(both):
    d = wrap180(both.efd_rotator_angle - both.physical_rotator_angle)
    print(f'visit1_efd agreement (n={len(both)})   : '
          f'mean {np.mean(d):+.4f} deg, std {np.std(d):.4f} deg')

top = missing_rate_table(vis, 'img_type').head(3)
if len(top):
    print('\nimg_type contributing the most missing visits:')
    for it, r in top.iterrows():
        print(f'  {str(it):<16} {int(r.missing):6d} missing of {int(r.n):6d} '
              f'({r.miss_frac * 100:.1f}%)')

if output_file:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    out = Path(output_dir) / output_file
    cols = ['visit_id', 'day_obs', 'seq_num', 'img_type', 'science_program',
            'band', 'rot_status', 'physical_rotator_angle',
            'efd_rotator_angle', 'rotator_best', 'rotator_best_source']
    vis.loc[vis.rot_status != 'present', [c for c in cols if c in vis.columns]] \
       .to_parquet(out, index=False)
    print(f'\nWrote {out}  ({n_miss} missing visits)')
